# Init

In [ ]:
# init
from importlib import reload
from pprint import pprint
import os
import sys
from pathlib import Path
import pandas as pd

def reloadPckg():
    reload(tgm)
    reload(too)
    reload(osmt)
    reload(tph)

sys.path.append('..')
sys.path.append('../tools')
devPath = "C:/Users/gonta/D/software developer/"
sys.path.append(os.path.join(devPath, "packages"))


import tests.osm_test as osmt
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph

# Cleaning

## * import

In [3]:
data_dir = Path(os.path.join(os.getcwd(), '..', 'data', 'raw', 'osm countries queries'))
files_dirs = [f for f in data_dir.glob('*')]
print(len(files_dirs))

raw_by_cntr = {}

# for chunks and non chunk files
for dir in files_dirs:
    files_elements = [tgm.load(str(f))['elements'] for f in dir.glob('*.json')]
    elements = [ele for list in files_elements for ele in list]
    raw_by_cntr[str(dir.name)] = elements

print(len(raw_by_cntr))

84
84


In [26]:
temp = [ele['id'] for ele in raw_by_cntr['UnitedStates'] if ele['tags']['admin_level'] == '6']
print(len(temp))

3380


In [10]:
state = tgm.load(r"C:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\UnitedStates\state.pkl")

In [25]:
print(len(state['6']['discovered']))

3375


In [18]:
print(len(state['6']['processed']))

40


In [16]:
print(len(raw_by_cntr))
path = Path(os.path.join(os.getcwd(),'../data/normalized'))
processsed_countries = [f.name for f in path.glob('*')]
print(len(processsed_countries))
# to_process_by_cntr = {k:v for k,v in raw_by_cntr.items() if k not in processsed_countries}
# print(len(to_process_by_cntr))

71
70


In [13]:
resume = {}
for country, raw in raw_by_cntr.items():
    resume[country] = {}
    resume[country]['lvl_1'] = len([ele for ele in raw if ele['tags']['admin_level'] == '4'])
    resume[country]['lvl_2'] = len([ele for ele in raw if ele['tags']['admin_level'] == '6'])
    resume[country]['lvl_3'] = len([ele for ele in raw if ele['tags']['admin_level'] == '8'])
    resume[country]['total'] = len([ele for ele in raw if ele['tags']['admin_level'] != '2'])

resume = pd.DataFrame(resume)
table = resume.T
tph.df_peek(table)

,lvl_1,lvl_2,lvl_3,total
Afghanistan,34,0,0,34
AkrotiriAndDhekelia,0,0,0,0
Albania,3,12,374,389
Algeria,58,547,1542,2147
Andorra,0,0,0,0
Angola,19,29,15,63
Anguilla,0,0,0,0
AntiguaAndBarbuda,3,8,0,11
Argentina,25,661,524,1210
Armenia,11,15,0,26


## * use sovereign countries only

In [14]:
sovereign_countries = tgm.load(os.path.join('..', 'data', 'sovereign countries.json'))
len(sovereign_countries)

214

In [15]:
print(len(raw_by_cntr))
raw_by_cntr = {k:df for k,df in raw_by_cntr.items() if k in sovereign_countries}
len(raw_by_cntr)

84


71

## * clean countries

In [17]:
import copy
def clean_country_data(raw_by_cntr):

    cleaned_by_cntr = copy.deepcopy(raw_by_cntr)  

    for country, raw_data in cleaned_by_cntr.items():

        # convert id to string
        for ele in raw_data:
            ele['id'] = str(ele['id'])

        # add parent_id to level 4 entities
        cntr_id = list(filter(lambda ele: ele['tags']['admin_level'] == '2', raw_data))[0]['id']
        for ele in raw_data:
            if ele['tags']['admin_level'] == '4':
                ele['tags']['parent_id'] = str(cntr_id)

        # add country_name and country_id tag
        for ele in raw_data:
            ele['tags']['country_name'] = country
            ele['tags']['country_id'] = cntr_id

    return cleaned_by_cntr

In [18]:
cleaned_by_cntr = clean_country_data(raw_by_cntr)
print(len(cleaned_by_cntr))

71


In [19]:
temp = [ele['tags'].get('parent_id',None) for k,v in cleaned_by_cntr.items() for ele in v]
pd.Series(temp).map(type).value_counts()

<class 'str'>         152920
<class 'NoneType'>        71
Name: count, dtype: int64

## * convert to dataframe

In [20]:
df_by_cntr = {k:too.normalizeOSM(elems) for k,elems in cleaned_by_cntr.items()}
len(df_by_cntr)

71

In [21]:
combined = pd.concat(df_by_cntr.values(), ignore_index=True)
combined.map(type).stack().value_counts()

<class 'pandas._libs.missing.NAType'>    1353320
<class 'str'>                            1094536
Name: count, dtype: int64

## * save

In [22]:
print(len(df_by_cntr))

71


In [23]:
# save data by country
for k,df in df_by_cntr.items():
    path = os.path.join('../data/normalized', k, f'{k}_normalized.pkl')
    tgm.dump(path, df)

## * review

In [24]:
# import
dataDir = Path(os.path.join(os.getcwd(), '..', 'data/normalized'))
rawFiles = [f for f in dataDir.rglob('*.pkl')]
print(len(rawFiles))

df_by_cntr = {
    file.parent.name:tgm.load(str(file))
    for file in rawFiles
}
print(len(df_by_cntr))

71
71


In [25]:
for k,df in df_by_cntr.items():
    cntr = df[df['tags.admin_level']=='2']
    print([k, cntr['tags.name'].iloc[0], cntr['tags.name:en'].iloc[0]])

['Afghanistan', 'افغانستان', 'Afghanistan']
['Albania', 'Shqipëria', 'Albania']
['Algeria', 'Algérie ⵍⵣⵣⴰⵢⴻⵔ الجزائر', 'Algeria']
['Andorra', 'Andorra', 'Andorra']
['Angola', 'Angola', 'Angola']
['Anguilla', 'Anguilla', 'Anguilla']
['AntiguaAndBarbuda', 'Antigua and Barbuda', 'Antigua and Barbuda']
['Argentina', 'Argentina', 'Argentina']
['Armenia', 'Հայաստան', 'Armenia']
['Australia', 'Australia', 'Australia']
['Austria', 'Österreich', 'Austria']
['Bangladesh', 'বাংলাদেশ', 'Bangladesh']
['Belarus', 'Беларусь', 'Belarus']
['Belgium', 'België / Belgique / Belgien', 'Belgium']
['Bhutan', 'འབྲུག་ཡུལ།', 'Bhutan']
['Brazil', 'Brasil', 'Brazil']
['Bulgaria', 'България', 'Bulgaria']
['Cambodia', 'ព្រះរាជាណាចក្រ\u200bកម្ពុជា', 'Cambodia']
['Chile', 'Chile', 'Chile']
['Colombia', 'Colombia', 'Colombia']
['Czechia', 'Česko', 'Czechia']
['Denmark', 'Danmark', 'Denmark']
['Ecuador', 'Ecuador', 'Ecuador']
['Estonia', 'Eesti', 'Estonia']
['Eswatini', 'Eswatini', 'Eswatini']
['FaroeIslands', 'Føroyar

In [ ]:
resume = {}
for country, df in df_by_cntr.items():
    resume[country] = {}
    resume[country]['lvl1'] = len(df[df['tags.admin_level'] == '4'])
    resume[country]['lvl2'] = len(df[df['tags.admin_level'] == '6'])
    resume[country]['lvl3'] = len(df[df['tags.admin_level'] == '8'])
    resume[country]['total'] = len(df[df['tags.admin_level'] != '2'])

resume = pd.DataFrame(resume)
table = resume.T
tph.df_peek(table)

,lvl1,lvl2,lvl3,total
Afghanistan,34,0,0,34
Albania,3,12,374,389
Algeria,58,547,1542,2147
Andorra,0,0,0,0
Angola,19,29,15,63
Anguilla,0,0,0,0
AntiguaAndBarbuda,3,8,0,11
Argentina,25,661,524,1210
Armenia,11,15,0,26
Australia,15,600,0,615


In [30]:
country_df = df_by_cntr['UnitedStates']
country_df[country_df['tags.admin_level'] == '4']['tags.name'].to_list()

['Vermont',
 'Massachusetts',
 'New York',
 'Maine',
 'New Hampshire',
 'New Brunswick / Nouveau-Brunswick',
 'Texas',
 'Illinois',
 'Missouri',
 'Kansas',
 'Oklahoma',
 'Arkansas',
 'Nebraska',
 'Iowa',
 'South Dakota',
 'North Dakota',
 'Kentucky',
 'Indiana',
 'Tennessee',
 'Mississippi',
 'Alabama',
 'Georgia',
 'Colorado',
 'Wyoming',
 'Utah',
 'New Mexico',
 'Arizona',
 'Florida',
 'Ohio',
 'West Virginia',
 'District of Columbia',
 'Pennsylvania',
 'Delaware',
 'Maryland',
 'Montana',
 'Idaho',
 'Wisconsin',
 'Minnesota',
 'Nevada',
 'California',
 'Oregon',
 'Washington',
 'Michigan',
 'Connecticut',
 'Hawaii',
 'South Carolina',
 'Virginia',
 'North Carolina',
 'Louisiana',
 'New Jersey',
 'United States Virgin Islands',
 'Guam',
 'Northern Mariana Islands',
 'British Columbia',
 'Yukon',
 'Rhode Island',
 'Alaska',
 'American Samoa',
 'Puerto Rico']